# मनुष्य-इन-द-लूप: प्री-अॅक्शन गेट्स, धोका स्तर निर्धारण, आणि ऑडिट लॉगिंग

या धडाचा README मनुष्य-इन-द-लूपची ओळख करुन देतो ज्यात एक छोटा स्निपेट असतो जो वापरकर्त्याला एजंटने आधीच प्रतिसाद दिल्यानंतर `APPROVE` किंवा `REJECT` विचारतो. तो नमुना सुरुवातीसाठी ठीक आहे, पण उत्पादनातील HITL अंमलबजावणी सामान्यतः आणखी तीन भागांची गरज असते:

1. एक **प्री-अॅक्शन गेट** जो एजंट जोखमीच्या टप्प्याला **पूर्वीच** चालवतो, ज्यामुळे खर्च, अपरिवर्तनीयता आणि विलंब नियंत्रणात राहतात.
2. **जोखीम स्तर निर्धारण**, ज्यामुळे कमी जोखमीच्या कृती आपोआप चालतात, मध्यम जोखमीच्या कृती बॅच-मनजूर केल्या जातात, आणि फक्त उच्च जोखमीच्या कृती मनुष्यावर अवलंबून राहतात.
3. एक **ऑडिट लॉग आणि पुनरावृत्ती लूप**, ज्यामुळे प्रत्येक गेट निर्णय JSONL म्हणून नोंदवला जातो, आणि नकार दिल्यास एजंटला फक्त `Revising...` छापण्याऐवजी संरचित कारणांसह पुनःप्रेरित केले जाते.

हा नोटबुक `06-system-message-framework.ipynb` मधील समान मूलतत्त्वांवर प्रत्येक बांधणी करतो. हा पूर्णपणे `DEMO_MODE = True` मध्ये (कोणतेही इंटरॅक्टिव्ह इनपुट आवश्यक नाही) किंवा `DEMO_MODE = False` मध्ये खऱ्या `input()` प्रॉम्प्टसह चालतो. टीप: DEMO_MODE मध्ये तिसऱ्या उद्दिष्टाचा पुनःप्रयत्न स्क्रिप्ट केलेला आहे त्यामुळे लूपची यांत्रिकी पूर्णपणे दिसते. खराखुरा पुनरावलोकन-चालित पुनर्वर्गीकरणासाठी `DEMO_MODE = False` आणि एखादा ऑपरेटर आवश्यक आहे.

**आउट ऑफ स्कोप (इतर धडात हाताळलेले):** प्रमाणीकरण आणि प्रवेश नियंत्रण (धडा 06 README धोका 2), टूल-कॉल मिडलवेअर (धडा 14 MAF सखोल अभ्यास), बहु-एजंट वादविवाद नमुने.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## नमुना 1: पूर्व-कृती गेट

README मधील HITL स्निपेट एजंटला प्रथम कॉल करते, नंतर वापरकर्त्याकडे आउटपुट मंजूर करण्यासाठी विचारले जाते. तो एक **पोस्ट-अॅक्शन** प्रवाह आहे. एजंट आधीच कार्यान्वित झाला आहे, त्यामुळे LLM कॉल खर्च आधीच दिला गेला आहे, आणि कोणताही साइड इफेक्ट (पाठवलेला ईमेल, लिहिलेली डेटाबेस रेकॉर्ड, पोस्ट केलेला टिप्पणी) आधीच झालेले आहे.

एक **पूर्व-अॅक्शन** प्रवाह गेट एजंटने धोकादायक टप्पा चालवण्यापूर्वी समाविष्ट करतो. एजंट क्रिया प्रस्तावित करतो, गेट कार्यान्वित करायचे की नाही ठरवतो, आणि केवळ मंजुरीवरच साइड इफेक्ट घडतो.

| पैलू | पोस्ट-अॅक्शन मंजुरी (README स्निपेट) | पूर्व-अॅक्शन गेट (हा नोटबुक) |
|---|---|---|
| मंजुरी कधी चालते? | एजंटने आउटपुट तयार केल्यानंतर | कोणताही साइड इफेक्ट होण्यापूर्वी |
| नाकारल्यावर LLM खर्च | आधीच दिला गेला आहे | केवळ प्रस्तावासाठी दिला जातो, क्रियेसाठी नाही |
| अपरिवर्तनीय साइड इफेक्ट | शक्य आहे (क्रिया आधीच झाली आहे) | प्रतिबंधित |
| लेखापरीक्षा स्पष्टता | मंजुरी हा प्रिंट स्टेटमेंट आहे | मंजुरी हा टाइमस्टॅम्प, कृती, कारण यासह JSON रेकॉर्ड आहे |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## नमुना 2: जोखमीचे स्तर निर्धारण

प्रत्येक क्रियेस मानवी मान्यता आवश्यक नसते. सार्वजनिक API विरुद्ध रीड-ओन्ली लुकअप याला ग्राहकाला ईमेल पाठवण्याच्या तुलनेत वेगळे धोके असतात. दोघांनाही एकसारखे वागताना ऑपरेटरचे लक्ष व्यर्थ जाते आणि एजंट मंदावतो.

एक सोपा 3-स्तरीय मॉडेल:

| स्तर | उदाहरणे | मान्यता प्रवाह |
|---|---|---|
| `कमी` (रीड-ओन्ली) | ज्ञानाधार शोधा, फ्लाइट पर्याय शोधा, सार्वजनिक वेब पृष्ठ प्राप्त करा | ऑटो-एक्झिक्यूट करा, लेखा परीक्षणासाठी नोंदविलेले |
| `मध्यम` (स्वस्त बदल) | निकाल कॅश करा, काउंटर वाढवा, अनुस्मारक नियोजित करा | ऑटो-एक्झिक्यूट करा, परंतु दैनिक पुनरावलोकनासाठी संकलित |
| `उच्च` (बाह्य समोरासमोर किंवा अपरिवर्तनीय) | ईमेल पाठवा, कार्ड चार्ज करा, सार्वजनिक चॅनेलवर पोस्ट करा | मानवी मान्यतेवर अवरोधित करा |

हे एक स्तर निर्धारण आहे. उत्पादन प्रणाली अनेकदा अधिक सूक्ष्म स्तर वापरतात (उदा. AWS IAM परवानगी स्तर, भूमिका-आधारित प्रवेश स्तर). खालील 3-स्तरीय आवृत्ती रीड-ओन्ली आणि बाजू परिणामकारक क्रिया मिश्रित करणाऱ्या एजंटसाठी सर्वात लहान उपयुक्त आवृत्ती आहे.

खालील वर्गीकरण कीवर्ड युक्ती वापरते ज्यामुळे डेमो ठराविक आणि स्वस्त राहतो. उत्पादन प्रणालीमध्ये तुम्ही याला शिकलेल्या वर्गीकरण करणाऱ्याने किंवा धोरण इंजिनने बदलाल.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## नमुना 3: लेखा परीक्षक लॉग आणि पुनरावलोकन लूप

`print("Response approved.")` हा लेखा परीक्षक लॉग नाही. विश्वासासाठी, प्रत्येक गेट निर्णय संरचित घटने म्हणून रेकॉर्ड केला पाहिजे, ज्याला नंतर तुम्ही क्वेरी करू शकता, पुन्हा चालवू शकता, किंवा एखाद्या घटना पुनरावलोकनाला संलग्न करू शकता.

दोन भाग:

1. **फक्त जोडणीयोग्य JSONL.** प्रत्येक निर्णयासाठी एक ओळ, ज्यात टाइमस्टॅम्प, क्रिया, टियर, निर्णय, कारण असते. ग्रीप करायला सोपे, नंतर प्रत्यक्ष लॉग स्टोअरमध्ये पाठवायला सोपे.
2. **नकारावर पुनरावलोकन लूप.** जेव्हा गेट `deny` परत करते, तेव्हा एजंट नकार कारणानुसार स्वतःला पुन्हा प्रॉम्प्ट करतो, जेणेकरून पुढील प्रस्ताव समस्येवर टाळू शकेल.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## अतिरिक्त साधने

अनेक इतर सार्वजनिक प्रकल्प हे HITL नमुन्यांचे विविध प्रकार अमलात आणतात. आपला स्टॅकमध्ये काय बसते हे शोधण्यासाठी पद्धतींची तुलना करा:

- **LangChain** मानव-इन-दी-लूप टूल रॅपर ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): मानवी इनपुटसाठी कार्यप्रवृत्ती थांबवणारे ड्रॉप-इन टूल रॅपर.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ यांनी याची रचना बदलली): बहु-एजंट संवादांमध्ये मानवी भूमिका सादर करण्यासाठी विशिष्ट एजंट भूमिका वापरतो.
- **Microsoft Agent Framework (MAF)** फंक्शन-इन्कवोकेशन मिडलवेअर ([docs](https://learn.microsoft.com/agent-framework/)): प्रत्येक टूल/फंक्शन कॉलच्या भोवती चालणारे मिडलवेअर, गेटिंग लॉजिक आणि मंजुरी प्रक्रियांसाठी योग्य.

प्रत्येक प्रकल्प तीन उप-नुमने वेगळ्या पद्धतीने हाताळतो: LangChain त्यांना टूल्स म्हणून रॅप करतो, AutoGen एजंट भूमिका वापरतो, आणि Microsoft Agent Framework फंक्शन-इन्कवोकेशन मिडलवेअर वापरतो. आपला स्वतःचा एजंट तयार करण्यापूर्वी एक किंवा दोन अंमलबजावणी पूर्णपणे वाचा.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
